# SupplyForge AI 2.0: KAN-31 - Categorical Features Encoding



Tài liệu này thực hiện việc mã hóa các biến phân loại của sản phẩm và kho hàng thành các biểu diễn tương thích với mô hình học máy (LightGBM) mà không gây rò rỉ dữ liệu.



Các biến phân loại gồm: `size`, `type`, `brand`, `categoryId`, và `warehouse` được mã hóa thành các số nguyên deterministic (Label Encoding) hoặc kiểu dữ liệu `category` của Pandas hỗ trợ nguyên bản thuật toán LightGBM.


### Quy tắc nghiệp vụ và Tiêu chuẩn Nghiệm thu (Acceptance Criteria):

1. **Mã hóa deterministic**: Xây dựng từ điển ánh xạ cho từng đặc trưng. Giá trị giống nhau luôn nhận cùng một mã số nguyên.

2. **Hỗ trợ nhãn ngoại lệ (Unknown Handling)**: Mỗi từ điển bổ sung nhãn đặc biệt `"Unknown"`. Khi gặp giá trị lạ trong quá trình inference, tự động gán về nhãn này.

3. **Xuất từ điển**: Lưu các ánh xạ thành các tệp JSON độc lập để tái sử dụng trong quá trình serving.

4. **Cập nhật dữ liệu**: Lưu trữ cột đã mã hóa vào danh mục tã bỉm `diaper_skus_attributes.csv` và lưới ngày `calendar_daily_sales_metadata.csv` (chuyển đổi thành kiểu `category`).


In [1]:
import pandas as pd

import numpy as np

import json

import re

import os



DATA_DIR = '/home/nguyen-dai-tung/supplyforge-ai/test-data/Database - NCKH'

order_path = os.path.join(DATA_DIR, 'order_1_cleaned.csv')

product_path = os.path.join(DATA_DIR, 'products_cleaned.json')

diaper_path = os.path.join(DATA_DIR, 'diaper_skus_attributes.csv')

grid_path = os.path.join(DATA_DIR, 'calendar_daily_sales_metadata.csv')



print("Đang tải các tập dữ liệu sạch...")

df_orders = pd.read_csv(order_path, low_memory=False)

df_products = pd.read_json(product_path)

df_diapers = pd.read_csv(diaper_path)

df_grid = pd.read_csv(grid_path)



print(f"Tổng số đơn hàng sạch: {len(df_orders)}")

print(f"Tổng số sản phẩm trong Master: {len(df_products)}")

print(f"Tổng số dòng trong lưới ngày: {len(df_grid)}")


Đang tải các tập dữ liệu sạch...


Tổng số đơn hàng sạch: 39369
Tổng số sản phẩm trong Master: 1335
Tổng số dòng trong lưới ngày: 323250


In [2]:
print("=== XÂY DỰNG BẢNG ÁNH XẠ KHO HÀNG (WAREHOUSE) ===")

# Ánh xạ kho hàng cho từng itemId từ lịch sử đơn hàng

print("Đang xây dựng ánh xạ kho hàng từ lịch sử đơn hàng sạch...")

item_wh_mapping = {}

for idx, row in df_orders.iterrows():

    wh = row.get('warehouse')

    if pd.notna(wh):

        try:

            data_dict = json.loads(row['data'])

            for item in data_dict.get('item_list', []):

                item_id = str(item.get('item_id', ''))

                if item_id:

                    item_wh_mapping[item_id] = wh

        except:

            pass



# Kiểm tra số lượng itemId có kho hàng

print(f"Số lượng itemId đã ánh xạ kho hàng: {len(item_wh_mapping)}")


=== XÂY DỰNG BẢNG ÁNH XẠ KHO HÀNG (WAREHOUSE) ===
Đang xây dựng ánh xạ kho hàng từ lịch sử đơn hàng sạch...


Số lượng itemId đã ánh xạ kho hàng: 346


In [3]:
print("=== TẠO TỪ ĐIỂN MÃ HÓA DETERMINISTIC CATEGORICAL ENCODING ===")



# Xác định các giá trị độc nhất từ nguồn dữ liệu rộng nhất có thể

categorical_cols_values = {

    'size': sorted(list(df_diapers['size'].dropna().astype(str).unique())),

    'type': sorted(list(df_diapers['type'].dropna().astype(str).unique())),

    'brand': sorted(list(df_products['brand'].dropna().astype(str).unique())),

    'categoryId': sorted([str(c) for c in df_products['categoryId'].dropna().unique()]),

    'warehouse': sorted(list(set(item_wh_mapping.values())))

}



encoding_dicts = {}

for col, vals in categorical_cols_values.items():

    # Bổ sung nhãn đặc biệt 'Unknown' để phục vụ inference an toàn

    if 'Unknown' not in vals:

        vals.append('Unknown')

        

    # Tạo ánh xạ deterministic tăng dần từ 0

    mapping = {val: idx for idx, val in enumerate(vals)}

    encoding_dicts[col] = mapping

    

    # Xuất từ điển ra file cấu hình JSON phục vụ downstream model serving

    json_path = os.path.join(DATA_DIR, f"{col}_mapping.json")

    with open(json_path, 'w', encoding='utf-8') as f:

        json.dump(mapping, f, indent=4, ensure_ascii=False)

    print(f"Đã xuất từ điển mã hóa {col} tại: {json_path}")


=== TẠO TỪ ĐIỂN MÃ HÓA DETERMINISTIC CATEGORICAL ENCODING ===
Đã xuất từ điển mã hóa size tại: /home/nguyen-dai-tung/supplyforge-ai/test-data/Database - NCKH/size_mapping.json
Đã xuất từ điển mã hóa type tại: /home/nguyen-dai-tung/supplyforge-ai/test-data/Database - NCKH/type_mapping.json
Đã xuất từ điển mã hóa brand tại: /home/nguyen-dai-tung/supplyforge-ai/test-data/Database - NCKH/brand_mapping.json
Đã xuất từ điển mã hóa categoryId tại: /home/nguyen-dai-tung/supplyforge-ai/test-data/Database - NCKH/categoryId_mapping.json
Đã xuất từ điển mã hóa warehouse tại: /home/nguyen-dai-tung/supplyforge-ai/test-data/Database - NCKH/warehouse_mapping.json


In [4]:
print("=== ÁP DỤNG MÃ HÓA LÊN TẬP THUỘC TÍNH TÃ BỈM ===")



# 1. Đảm bảo ép kiểu itemId làm key ghép nối chính xác

df_diapers['itemId'] = df_diapers['itemId'].astype(str)

df_products['itemId'] = df_products['itemId'].astype(str)



# 2. Ghép nối các trường brand và categoryId nếu chưa có trong diaper attributes

df_prod_metadata = df_products[['itemId', 'brand', 'categoryId']].drop_duplicates(subset=['itemId'])



# Xóa cột trùng trước khi merge

for c in ['brand', 'categoryId']:

    if c in df_diapers.columns:

        df_diapers = df_diapers.drop(columns=[c])

        

df_diapers = pd.merge(df_diapers, df_prod_metadata, on='itemId', how='left')



# Điền khuyết metadata

df_diaper_wh = df_diapers['itemId'].map(item_wh_mapping).fillna('Kho HVN')

df_diapers['warehouse'] = df_diaper_wh

df_diapers['brand'] = df_diapers['brand'].fillna('unknown_brand')

df_diapers['categoryId'] = df_diapers['categoryId'].fillna('unknown_category')



# 3. Áp dụng mã hóa số hóa

categorical_cols = ['size', 'type', 'brand', 'categoryId', 'warehouse']

for col in categorical_cols:

    mapping = encoding_dicts[col]

    unknown_code = mapping['Unknown']

    df_diapers[f"{col}_encoded"] = df_diapers[col].apply(lambda x: mapping.get(str(x), unknown_code))



# Lưu dữ liệu thuộc tính tã bỉm đã cập nhật mã hóa

df_diapers.to_csv(diaper_path, index=False)

print(f"Đã lưu thuộc tính tã bỉm đã mã hóa tại: {diaper_path}")


=== ÁP DỤNG MÃ HÓA LÊN TẬP THUỘC TÍNH TÃ BỈM ===
Đã lưu thuộc tính tã bỉm đã mã hóa tại: /home/nguyen-dai-tung/supplyforge-ai/test-data/Database - NCKH/diaper_skus_attributes.csv


In [5]:
print("=== CẬP NHẬT LƯỚI THỜI GIAN DAILY CALENDAR GRID ===")



# 1. Đồng nhất kiểu dữ liệu khóa itemId trên lưới

df_grid['itemId'] = df_grid['itemId'].astype(str)



# 2. Ánh xạ kho hàng (warehouse) cho toàn bộ lưới thời gian ngày

df_grid['warehouse'] = df_grid['itemId'].map(item_wh_mapping).fillna('Kho HVN')



# 3. Áp dụng mã hóa brand, categoryId, warehouse vào lưới ngày sử dụng từ điển cố định

cols_to_encode_grid = ['brand', 'categoryId', 'warehouse']

for col in cols_to_encode_grid:

    mapping = encoding_dicts[col]

    unknown_code = mapping['Unknown']

    df_grid[f"{col}_encoded"] = df_grid[col].apply(lambda x: mapping.get(str(x), unknown_code))



# 4. Đảm bảo chuyển đổi các đặc trưng phân loại thô thành kiểu category của Pandas hỗ trợ LightGBM

for col in cols_to_encode_grid:

    df_grid[f"{col}_encoded"] = df_grid[f"{col}_encoded"].astype('category')



# 5. Kiểm tra Null và tính toàn vẹn số dòng của lưới thời gian ngày

assert len(df_grid) == 323250, f"Lỗi: Số dòng lưới bị thay đổi ({len(df_grid)} dòng)!"

for col in cols_to_encode_grid:

    assert df_grid[f"{col}_encoded"].isnull().sum() == 0, f"Lỗi: Có giá trị null ở cột {col}_encoded!"



# 6. Lưu đè lưới thời gian đã làm giàu đặc trưng mã hóa

df_grid.to_csv(grid_path, index=False)

print(f"Đã cập nhật thành công lưới ngày tích hợp mã hóa tại: {grid_path}")


=== CẬP NHẬT LƯỚI THỜI GIAN DAILY CALENDAR GRID ===


Đã cập nhật thành công lưới ngày tích hợp mã hóa tại: /home/nguyen-dai-tung/supplyforge-ai/test-data/Database - NCKH/calendar_daily_sales_metadata.csv


In [6]:
print("=== THỰC THI KIỂM CHỨNG & XÁC THỰC (VERIFICATION) ===\n")



total_skus = len(df_diapers)

null_sizes = df_diapers['size'].isnull().sum()

null_types = df_diapers['type'].isnull().sum()



# Đảm bảo không có giá trị khuyết thiếu trong các cột mã hóa mới sinh ra

for col in ['size', 'type', 'brand', 'categoryId', 'warehouse']:

    assert df_diapers[f"{col}_encoded"].isnull().sum() == 0, f"Lỗi: Phát hiện Null ở đặc trưng mã hóa {col}!"

print("1. Đảm bảo các cột đặc trưng mã hóa mới không chứa giá trị Null: Đạt")



# Đảm bảo các giá trị số nguyên mã hóa là deterministic và hợp lệ

for col in ['size', 'type', 'brand', 'categoryId', 'warehouse']:

    mapping = encoding_dicts[col]

    assert df_diapers[f"{col}_encoded"].isin(mapping.values()).all(), f"Lỗi: Giá trị mã hóa của {col} không hợp lệ!"

print("2. Tính hợp lệ của mã hóa số nguyên: Đạt")



# Đảm bảo định dạng kiểu dữ liệu trên lưới hỗ trợ nguyên bản LightGBM

for col in ['brand', 'categoryId', 'warehouse']:

    assert df_grid[f"{col}_encoded"].dtype.name == 'category', f"Lỗi: Cột {col}_encoded không phải kiểu category!"

print("3. Định dạng Pandas category dtype tương thích với LightGBM: Đạt")



# In ngẫu nhiên 30 bản ghi thuộc tính tã bỉm để kiểm thử thủ công

print("\n4. In ngẫu nhiên 30 SKU đã mã hóa số hóa để kiểm định trực quan:")

print(df_diapers[['skuName', 'size', 'size_encoded', 'type', 'type_encoded', 'brand', 'brand_encoded', 'warehouse', 'warehouse_encoded']].sample(30, random_state=42).to_string())


=== THỰC THI KIỂM CHỨNG & XÁC THỰC (VERIFICATION) ===

1. Đảm bảo các cột đặc trưng mã hóa mới không chứa giá trị Null: Đạt
2. Tính hợp lệ của mã hóa số nguyên: Đạt
3. Định dạng Pandas category dtype tương thích với LightGBM: Đạt

4. In ngẫu nhiên 30 SKU đã mã hóa số hóa để kiểm định trực quan:
                                    skuName size  size_encoded   type  type_encoded    brand  brand_encoded warehouse  warehouse_encoded
126              Supdry Xanh,L quần - 1 cái    L             0  PANTS             0   Supdry             65   Kho HVN                  0
45   Cộng Miếng (Xanh),Nữa bịch NB dùng thử   NB             2  PANTS             0    Moony             49   Kho HVN                  0
133              Supdry Trắng,S dán - 1 cái    S             3   TAPE             1   Supdry             65   Kho HVN                  0
138                            Tã Dán - S82    S             3   TAPE             1  Huggies             35   Kho HVN                  0
111                